# 🤖 DLavie OS — LoRA Fine-Tuning Notebook

Notebook ini melakukan **real LoRA fine-tuning** menggunakan dataset dari DLavie OS.

### Sebelum mulai:
1. Pastikan **GPU aktif**: Settings → Accelerator → GPU T4 x2
2. Upload file `dataset.jsonl` dari DLavie OS (Training Hub → Export)
3. Jalankan cell satu per satu dengan **Shift+Enter**

In [ ]:
# Cell 1 — Install semua library yang dibutuhkan
!pip install -q transformers peft trl accelerate bitsandbytes datasets sentencepiece
print('✅ Library terinstall')

In [ ]:
# Cell 2 — Cek GPU
import torch
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Nama GPU   :', torch.cuda.get_device_name(0))
    print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️ GPU tidak aktif! Aktifkan di Settings → Accelerator → GPU T4 x2')

In [ ]:
# Cell 3 — Konfigurasi (edit sesuai kebutuhan)

# Model base yang akan di-fine-tune
# Pilihan ringan: 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# Pilihan lebih bagus: 'Qwen/Qwen2.5-1.5B-Instruct'
BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Path dataset (ganti jika nama file berbeda)
DATASET_PATH = '/kaggle/input/dlavie-dataset/dataset.jsonl'

# Nama model output
OUTPUT_NAME = 'dlavie-custom-v1'

# Hyperparameter training
EPOCHS      = 3
LORA_RANK   = 16
BATCH_SIZE  = 4
LR          = 2e-4
MAX_SEQ_LEN = 512

print(f'✅ Config: {BASE_MODEL} | {EPOCHS} epochs | LoRA r={LORA_RANK}')

In [ ]:
# Cell 4 — Load & preview dataset
import json
import os

# Coba load dari path Kaggle, fallback ke direktori saat ini
if os.path.exists(DATASET_PATH):
    path = DATASET_PATH
else:
    # Cari di direktori kerja
    for f in os.listdir('.'):
        if f.endswith('.jsonl'):
            path = f
            break
    else:
        raise FileNotFoundError('dataset.jsonl tidak ditemukan. Upload file terlebih dahulu.')

samples = []
with open(path) as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

# Filter sample valid (harus ada input + output)
samples = [s for s in samples if s.get('input') and s.get('output')]

print(f'✅ Dataset dimuat: {len(samples)} sample valid')
print()
print('Contoh sample pertama:')
print('INPUT :', samples[0]['input'][:100], '...')
print('OUTPUT:', samples[0]['output'][:100], '...')

In [ ]:
# Cell 5 — Format dataset ke chat template
from datasets import Dataset

def format_sample(sample):
    return (
        f"### System:\nYou are DLavie OS, a powerful AI assistant.\n\n"
        f"### Human:\n{sample['input']}\n\n"
        f"### Assistant:\n{sample['output']}"
    )

texts = [format_sample(s) for s in samples]
dataset = Dataset.from_dict({'text': texts})

print(f'✅ Dataset diformat: {len(dataset)} baris')
print()
print('Contoh formatted:')
print(dataset[0]['text'][:300], '...')

In [ ]:
# Cell 6 — Load model + tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

print(f'Loading tokenizer: {BASE_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model dengan 4-bit quantization untuk hemat VRAM
print(f'Loading model (4-bit quantized) ...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Tambahkan LoRA adapters
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model siap untuk training!')

In [ ]:
# Cell 7 — TRAINING (ini yang sebenarnya!)
from trl import SFTTrainer, SFTConfig

print('🚀 Memulai training...')

training_args = SFTConfig(
    output_dir=f'./{OUTPUT_NAME}',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    max_seq_length=MAX_SEQ_LEN,
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()
print('✅ Training selesai!')

In [ ]:
# Cell 8 — Simpan model hasil training
import os

save_path = f'./{OUTPUT_NAME}-final'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Zip untuk mudah didownload
os.system(f'zip -r {OUTPUT_NAME}.zip {save_path}/')

print(f'✅ Model disimpan di: {save_path}')
print(f'✅ File ZIP: {OUTPUT_NAME}.zip')
print()
print('Untuk download: klik ikon folder di panel kiri → temukan file .zip → klik kanan → Download')

In [ ]:
# Cell 9 — Test model hasil training
from transformers import pipeline

print('Testing model hasil training...')
print()

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
)

test_prompt = '### System:\nYou are DLavie OS, a powerful AI assistant.\n\n### Human:\nHello! Who are you?\n\n### Assistant:\n'
result = pipe(test_prompt)

print('Prompt:', 'Hello! Who are you?')
print()
print('Response:', result[0]['generated_text'][len(test_prompt):])